# Multiclass Classifier Retrain on All Labeled Data

This Kaggle notebook rebuilds the supervised multiclass wafer-defect ensemble on **all currently labeled WM-811K rows** with a stratified **80 / 10 / 10** split.

Why this setup matters:
- the older `50k` classifier subset already used every labeled defect row, so it could not provide a true unseen-defect holdout
- this workflow keeps held-out examples from every labeled class in both validation and test
- the final evaluation reports multiclass metrics such as **accuracy**, **balanced accuracy**, **precision**, **recall**, and **F1**
- it also exports a binary `defect vs none` view using `1 - P(none)` for a report-style anomaly interpretation

Important comparison note:
- use the same metric family for consistency with `REPORT.md`
- do **not** merge these numbers directly into the older anomaly leaderboard, because the split is intentionally different


In [ ]:
from pathlib import Path
import os
import shutil
import sys


def resolve_unique_input_path(pattern: str, description: str) -> Path:
    matches = sorted(Path("/kaggle/input").glob(pattern))
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            f"Could not find {description} with pattern '/kaggle/input/{pattern}'. "
            "Attach the Kaggle dataset first or set SOURCE manually."
        )
    raise RuntimeError(
        f"Found multiple {description} matches: {matches}. "
        "Set SOURCE manually to the correct mounted path."
    )


SOURCE = None  # Optional manual override: Path('/kaggle/input/<dataset>/classifier_all_80_10_10_bundle')
if SOURCE is None:
    SOURCE = resolve_unique_input_path("*/classifier_all_80_10_10_bundle", "training bundle")
else:
    SOURCE = Path(SOURCE)

TARGET = Path("/kaggle/working/classifier_all_80_10_10_bundle")
if TARGET.exists():
    shutil.rmtree(TARGET)
shutil.copytree(SOURCE, TARGET)
os.chdir(TARGET)

SRC_ROOT = TARGET / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("Bundle source:", SOURCE)
print("Running from:", TARGET)


In [ ]:
from pathlib import Path
import json
import os
import subprocess

import pandas as pd
from IPython.display import display


def run_logged(title: str, command: list[str], env: dict[str, str], cwd: Path) -> None:
    print(f"\n{'=' * 80}\n{title}\n{'=' * 80}")
    print("Command:", " ".join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)
    print(f"Completed: {title}\n")


def resolve_unique_input_path(pattern: str, description: str) -> Path:
    matches = sorted(Path("/kaggle/input").glob(pattern))
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            f"Could not find {description} with pattern '/kaggle/input/{pattern}'. "
            "Attach the Kaggle dataset first or set RAW_PICKLE manually."
        )
    raise RuntimeError(
        f"Found multiple {description} matches: {matches}. "
        "Set RAW_PICKLE manually to the correct mounted path."
    )


REPO_ROOT = Path.cwd().resolve()
RAW_PICKLE = None  # Optional manual override: Path('/kaggle/input/<dataset>/LSWMD.pkl')
if RAW_PICKLE is None:
    RAW_PICKLE = resolve_unique_input_path("*/LSWMD.pkl", "raw WM-811K pickle")
else:
    RAW_PICKLE = Path(RAW_PICKLE)

print("Repo root:", REPO_ROOT)
print("Raw pickle:", RAW_PICKLE)


In [ ]:
data_config_source = REPO_ROOT / "configs/data/classifier/data_multiclass_all_80_10_10.toml"
runtime_dir = REPO_ROOT / "runtime_configs"
runtime_dir.mkdir(parents=True, exist_ok=True)
data_config_runtime = runtime_dir / "data_multiclass_all_80_10_10.kaggle.toml"

config_text = data_config_source.read_text(encoding="utf-8")
config_text = config_text.replace(
    'raw_pickle = "data/raw/LSWMD.pkl"',
    f'raw_pickle = "{RAW_PICKLE.as_posix()}"',
)
data_config_runtime.write_text(config_text, encoding="utf-8")

print("Runtime data config:", data_config_runtime)
print(data_config_runtime.read_text(encoding="utf-8"))


In [ ]:
prepare_command = [
    sys.executable,
    "-u",
    str(REPO_ROOT / "scripts/classifier/prepare_wm811k_multiclass.py"),
    "--config",
    str(data_config_runtime),
]
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "src")
env["PYTHONUNBUFFERED"] = "1"

run_logged("Prepare all-labeled 80/10/10 dataset", prepare_command, env=env, cwd=REPO_ROOT)


## Training Execution Cell

The next cell is the one that actually launches classifier training. It runs the training script once for each seed config and streams progress logs into the notebook output.


In [ ]:
summary_path = REPO_ROOT / "data/processed/x64/wm811k_multiclass_all_80_10_10/dataset_summary.txt"
metadata_path = REPO_ROOT / "data/processed/x64/wm811k_multiclass_all_80_10_10/metadata_labeled_all.csv"

print(summary_path.read_text(encoding="utf-8"))

metadata = pd.read_csv(metadata_path)
display(metadata.head())
display(metadata.groupby(["split", "label_name"]).size().unstack(fill_value=0))


In [ ]:
seed_configs = [
    REPO_ROOT / "configs/training/classifier/train_multiclass_classifier_all_80_10_10_seed07.toml",
    REPO_ROOT / "configs/training/classifier/train_multiclass_classifier_all_80_10_10_seed13.toml",
    REPO_ROOT / "configs/training/classifier/train_multiclass_classifier_all_80_10_10_seed21.toml",
]
epochs_override = None
batch_size_override = None

for config_path in seed_configs:
    command = [
        sys.executable,
        "-u",
        str(REPO_ROOT / "scripts/classifier/train_multiclass_classifier.py"),
        "--config",
        str(config_path),
        "--log-interval",
        "25",
    ]
    if epochs_override is not None:
        command.extend(["--epochs", str(epochs_override)])
    if batch_size_override is not None:
        command.extend(["--batch-size", str(batch_size_override)])

    run_logged(f"Train classifier seed config: {config_path.name}", command, env=env, cwd=REPO_ROOT)


In [ ]:
checkpoint_paths = [
    REPO_ROOT / "artifacts/multiclass_classifier_all_80_10_10_seed07/best_model.pt",
    REPO_ROOT / "artifacts/multiclass_classifier_all_80_10_10_seed13/best_model.pt",
    REPO_ROOT / "artifacts/multiclass_classifier_all_80_10_10_seed21/best_model.pt",
]
evaluation_output_dir = REPO_ROOT / "artifacts/multiclass_classifier_all_80_10_10_ensemble_eval"
ensemble_mode = "average"  # Change to 'stacking' if you want to fit a validation combiner.

evaluation_command = [
    sys.executable,
    "-u",
    str(REPO_ROOT / "scripts/classifier/evaluate_multiclass_classifier_metrics.py"),
    "--metadata-csv",
    str(metadata_path),
    "--output-dir",
    str(evaluation_output_dir),
    "--combiner",
    ensemble_mode,
    "--checkpoints",
    *[str(path) for path in checkpoint_paths],
]

run_logged("Evaluate ensemble on validation and test", evaluation_command, env=env, cwd=REPO_ROOT)


In [ ]:
metrics_path = evaluation_output_dir / "metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))

multiclass_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "accuracy": split_metrics["multiclass"]["accuracy"],
            "balanced_accuracy": split_metrics["multiclass"]["balanced_accuracy"],
            "macro_precision": split_metrics["multiclass"]["macro_precision"],
            "macro_recall": split_metrics["multiclass"]["macro_recall"],
            "macro_f1": split_metrics["multiclass"]["macro_f1"],
            "weighted_f1": split_metrics["multiclass"]["weighted_f1"],
        }
        for split_name, split_metrics in metrics["splits"].items()
    ]
)

binary_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "threshold": split_metrics["binary_defect_vs_none"]["threshold"],
            "accuracy": split_metrics["binary_defect_vs_none"]["accuracy"],
            "balanced_accuracy": split_metrics["binary_defect_vs_none"]["balanced_accuracy"],
            "precision": split_metrics["binary_defect_vs_none"]["precision"],
            "recall": split_metrics["binary_defect_vs_none"]["recall"],
            "f1": split_metrics["binary_defect_vs_none"]["f1"],
            "auroc": split_metrics["binary_defect_vs_none"]["auroc"],
            "auprc": split_metrics["binary_defect_vs_none"]["auprc"],
            "best_sweep_f1": split_metrics["binary_defect_vs_none"]["best_sweep_f1"],
        }
        for split_name, split_metrics in metrics["splits"].items()
    ]
)

per_class_report = pd.DataFrame(metrics["splits"]["test"]["multiclass"]["classification_report"]).T
per_defect_recall = pd.DataFrame(metrics["splits"]["test"]["binary_defect_vs_none"]["per_defect_recall"]).T

print("Combiner:", metrics["combiner"])
print("Validation threshold used for defect-vs-none view:", metrics["val_threshold_selection"]["threshold"])
display(multiclass_summary)
display(binary_summary)
display(per_class_report.loc[metrics["class_names"], ["precision", "recall", "f1-score", "support"]])
display(per_defect_recall)


## Next Steps

If this split beats the older `50k` ensemble on the metrics you care about, the clean next move is to treat this as the new main classifier training recipe and optionally add a second pass for pseudo-label generation later.
